## Invoice DocAI (v2) — 04 Robustness Evaluation

**Goal:** Compare OCR and Donut pipelines on messenger-grade corrupted images.

Corruptions: JPEG compression (q=22), Gaussian blur (k=5),
perspective warp (8%), downscale-upscale (0.6x).

In [ ]:
import sys, time, re, json
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

def is_project_root(p: Path) -> bool:
    return (p / 'data' / 'sroie').exists() and ((p / 'notebooks').exists() or (p / 'v2' / 'notebooks').exists())

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd, cwd.parent, cwd.parent.parent,
        cwd / 'invoice_docai',
        Path('/content/drive/MyDrive/ORC/invoice_docai'),
        Path('/content/drive/MyDrive/invoice_docai'),
        Path('/content/drive/MyDrive/From OCR 2 Transformers/invoice_docai'),
        Path(r'c:\Yandex.Disk\Yandex.Disk\ML Neimark\From OCR 2 Transformers\invoice_docai'),
    ]
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if is_project_root(p):
            return p
    raise FileNotFoundError('Cannot locate project root')

PROJECT_ROOT = resolve_project_root()
V2_SRC = PROJECT_ROOT / 'v2' / 'src'
if V2_SRC.exists() and str(V2_SRC) not in sys.path:
    sys.path.insert(0, str(V2_SRC))
elif (PROJECT_ROOT / 'src').exists():
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from docai_utils import (
    extract_fields_from_lines, messenger_corrupt,
    evaluate, normalize_date, normalize_total
)

PROCESSED = PROJECT_ROOT / 'data' / 'sroie' / 'processed'
OUTPUT_ROOT = PROJECT_ROOT / 'v2' / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CORRUPT_DIR = OUTPUT_ROOT / 'robustness'
CORRUPT_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(PROCESSED / 'manifest_val.csv').dropna(subset=['image_path']).reset_index(drop=True)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Val docs: {len(manifest)}')

## 1. Generate corrupted images

In [ ]:
import random
random.seed(42)
np.random.seed(42)

corrupt_paths = {}

for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc='Corrupting'):
    img_bgr = cv2.imread(row['image_path'])
    if img_bgr is None:
        print(f'WARNING: cannot read {row["image_path"]}')
        continue
    corrupted = messenger_corrupt(
        img_bgr,
        jpeg_quality=22, blur_kernel=5,
        perspective=0.08, downscale=0.6
    )
    out_path = CORRUPT_DIR / f'{row["id"]}_messenger.jpg'
    cv2.imwrite(str(out_path), corrupted)
    corrupt_paths[row['id']] = str(out_path)

print(f'Generated {len(corrupt_paths)} corrupted images')

## 2. Show examples: original vs corrupted

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
sample_ids = manifest.sample(4, random_state=42)['id'].tolist()

for i, doc_id in enumerate(sample_ids):
    row = manifest[manifest['id'] == doc_id].iloc[0]
    orig = Image.open(row['image_path'])
    corr = Image.open(corrupt_paths[doc_id])

    axes[0, i].imshow(orig)
    axes[0, i].set_title(f'Original: {doc_id}', fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(corr)
    axes[1, i].set_title('Corrupted (messenger)', fontsize=8)
    axes[1, i].axis('off')

plt.suptitle('Original vs Messenger-grade Corrupted', fontsize=14)
plt.tight_layout()
plt.show()

## 3. EasyOCR inference on corrupted images

In [ ]:
import easyocr

gpu_available = False
try:
    import torch
    gpu_available = torch.cuda.is_available()
except Exception:
    pass

reader = easyocr.Reader(['en'], gpu=gpu_available)
print(f'EasyOCR ready (GPU={gpu_available})')

In [ ]:
# Checkpointed OCR inference on corrupted images
ocr_corr_checkpoint = OUTPUT_ROOT / 'ocr_corrupted_checkpoint.csv'
done_ids_ocr = set()
ocr_corr_rows = []
ocr_corr_latencies = []

if ocr_corr_checkpoint.exists():
    done_df = pd.read_csv(ocr_corr_checkpoint)
    done_ids_ocr = set(done_df['id'].tolist())
    ocr_corr_rows = done_df.to_dict('records')
    print(f'Resuming OCR corrupted: {len(done_ids_ocr)} done')

for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc='OCR corrupted'):
    if row['id'] in done_ids_ocr or row['id'] not in corrupt_paths:
        continue

    t0 = time.perf_counter()
    ocr_result = reader.readtext(corrupt_paths[row['id']], detail=1, paragraph=False)
    lines = [item[1] for item in ocr_result]
    fields = extract_fields_from_lines(lines)
    elapsed = (time.perf_counter() - t0) * 1000

    ocr_corr_rows.append({
        'id': row['id'],
        'pred_vendor': fields['pred_vendor'],
        'pred_date': fields['pred_date'],
        'pred_total': fields['pred_total'],
    })
    ocr_corr_latencies.append(elapsed)

    if len(ocr_corr_rows) % 50 == 0:
        pd.DataFrame(ocr_corr_rows).to_csv(ocr_corr_checkpoint, index=False)

ocr_corr_df = pd.DataFrame(ocr_corr_rows)
ocr_corr_df.to_csv(OUTPUT_ROOT / 'ocr_predictions_corrupted.csv', index=False)
if ocr_corr_checkpoint.exists():
    ocr_corr_checkpoint.unlink()

print(f'Saved {len(ocr_corr_df)} corrupted OCR predictions')
if ocr_corr_latencies:
    print(f'Latency mean: {np.mean(ocr_corr_latencies):.0f} ms/doc')

## 4. Donut inference on corrupted images (optional)

In [ ]:
DONUT_AVAILABLE = False

try:
    import torch
    from transformers import DonutProcessor, VisionEncoderDecoderModel

    # Try fine-tuned first, then pre-trained
    ft_dir = PROJECT_ROOT / 'v2' / 'checkpoints' / 'donut-sroie-finetuned'
    if ft_dir.exists() and (ft_dir / 'config.json').exists():
        donut_processor = DonutProcessor.from_pretrained(str(ft_dir))
        donut_model = VisionEncoderDecoderModel.from_pretrained(str(ft_dir))
        DONUT_TASK_PROMPT = '<s_sroie>'
        print(f'Loaded fine-tuned Donut from {ft_dir}')
    else:
        donut_processor = DonutProcessor.from_pretrained('philschmid/donut-base-sroie')
        donut_model = VisionEncoderDecoderModel.from_pretrained('philschmid/donut-base-sroie')
        DONUT_TASK_PROMPT = '<s_sroie>'
        print('Loaded pre-trained philschmid/donut-base-sroie')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    donut_model = donut_model.to(device).eval()
    DONUT_AVAILABLE = True

except Exception as e:
    print(f'Donut not available: {e}')
    print('Skipping Donut corrupted inference.')

In [ ]:
if DONUT_AVAILABLE:
    def donut_predict_corrupted(image_path):
        image = Image.open(image_path).convert('RGB')
        pixel_values = donut_processor(image, return_tensors='pt').pixel_values.to(device)
        decoder_input_ids = donut_processor.tokenizer(
            DONUT_TASK_PROMPT, add_special_tokens=False, return_tensors='pt'
        ).input_ids.to(device)

        with torch.no_grad():
            outputs = donut_model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                max_length=donut_model.decoder.config.max_position_embeddings,
                pad_token_id=donut_processor.tokenizer.pad_token_id,
                eos_token_id=donut_processor.tokenizer.eos_token_id,
                num_beams=1, use_cache=True,
                bad_words_ids=[[donut_processor.tokenizer.unk_token_id]],
            )
        seq = donut_processor.batch_decode(outputs)[0]
        seq = seq.replace(donut_processor.tokenizer.eos_token, '').replace(donut_processor.tokenizer.pad_token, '')

        vendor = date = total = ''
        m = re.search(r'<s_company>(.*?)</s_company>', seq, re.DOTALL)
        if m: vendor = m.group(1).strip()
        m = re.search(r'<s_date>(.*?)</s_date>', seq, re.DOTALL)
        if m: date = normalize_date(m.group(1).strip())
        m = re.search(r'<s_total>(.*?)</s_total>', seq, re.DOTALL)
        if m: total = normalize_total(m.group(1).strip())

        return {'pred_vendor': vendor, 'pred_date': date, 'pred_total': total}

    donut_corr_rows = []
    donut_corr_latencies = []

    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc='Donut corrupted'):
        if row['id'] not in corrupt_paths:
            continue
        t0 = time.perf_counter()
        pred = donut_predict_corrupted(corrupt_paths[row['id']])
        elapsed = (time.perf_counter() - t0) * 1000
        donut_corr_rows.append({'id': row['id'], **pred})
        donut_corr_latencies.append(elapsed)

    donut_corr_df = pd.DataFrame(donut_corr_rows)
    donut_corr_df.to_csv(OUTPUT_ROOT / 'donut_predictions_corrupted.csv', index=False)
    print(f'Saved {len(donut_corr_df)} corrupted Donut predictions')
    if donut_corr_latencies:
        print(f'Latency mean: {np.mean(donut_corr_latencies):.0f} ms/doc')
else:
    print('Donut not available — skipping corrupted inference.')

## 5. Compute degradation: clean vs corrupted

In [ ]:
gt_cols = ['id', 'gt_vendor', 'gt_date', 'gt_total']
gt = manifest[gt_cols]

# Load clean predictions
results = []

# OCR clean
ocr_clean_path = OUTPUT_ROOT / 'ocr_predictions_clean.csv'
if ocr_clean_path.exists():
    ocr_clean_df = pd.read_csv(ocr_clean_path)
    m, _ = evaluate(gt, ocr_clean_df)
    for _, r in m.iterrows():
        results.append({'pipeline': 'EasyOCR', 'condition': 'Clean', 'field': r['field'],
                        'precision': r['precision'], 'recall': r['recall'], 'f1': r['f1'],
                        'exact_match': r.get('exact_match', np.nan)})
    print('Loaded OCR clean predictions')
else:
    print(f'WARNING: {ocr_clean_path} not found. Run 02_baseline_ocr first.')

# OCR corrupted
ocr_corr_path = OUTPUT_ROOT / 'ocr_predictions_corrupted.csv'
if ocr_corr_path.exists():
    ocr_corr_df = pd.read_csv(ocr_corr_path)
    m, _ = evaluate(gt, ocr_corr_df)
    for _, r in m.iterrows():
        results.append({'pipeline': 'EasyOCR', 'condition': 'Corrupted', 'field': r['field'],
                        'precision': r['precision'], 'recall': r['recall'], 'f1': r['f1'],
                        'exact_match': r.get('exact_match', np.nan)})

# Donut clean
donut_clean_path = OUTPUT_ROOT / 'donut_predictions_clean.csv'
if donut_clean_path.exists():
    donut_clean_df = pd.read_csv(donut_clean_path)
    m, _ = evaluate(gt, donut_clean_df)
    for _, r in m.iterrows():
        results.append({'pipeline': 'Donut', 'condition': 'Clean', 'field': r['field'],
                        'precision': r['precision'], 'recall': r['recall'], 'f1': r['f1'],
                        'exact_match': r.get('exact_match', np.nan)})
    print('Loaded Donut clean predictions')
else:
    print(f'WARNING: {donut_clean_path} not found. Run 03 or 03b first.')

# Donut corrupted
donut_corr_path = OUTPUT_ROOT / 'donut_predictions_corrupted.csv'
if donut_corr_path.exists():
    donut_corr_df = pd.read_csv(donut_corr_path)
    m, _ = evaluate(gt, donut_corr_df)
    for _, r in m.iterrows():
        results.append({'pipeline': 'Donut', 'condition': 'Corrupted', 'field': r['field'],
                        'precision': r['precision'], 'recall': r['recall'], 'f1': r['f1'],
                        'exact_match': r.get('exact_match', np.nan)})

results_df = pd.DataFrame(results)
display(results_df)

## 6. Degradation table

In [ ]:
# Compute degradation: F1 drop from Clean to Corrupted
degrad_rows = []

for pipeline in results_df['pipeline'].unique():
    for field in results_df['field'].unique():
        clean_row = results_df[(results_df['pipeline']==pipeline) & (results_df['condition']=='Clean') & (results_df['field']==field)]
        corr_row  = results_df[(results_df['pipeline']==pipeline) & (results_df['condition']=='Corrupted') & (results_df['field']==field)]

        if len(clean_row) == 0 or len(corr_row) == 0:
            continue

        f1_clean = clean_row.iloc[0]['f1']
        f1_corr  = corr_row.iloc[0]['f1']
        drop_pp = f1_clean - f1_corr
        drop_pct = (drop_pp / f1_clean * 100) if f1_clean > 0 else 0

        degrad_rows.append({
            'pipeline': pipeline,
            'field': field,
            'F1_clean': f1_clean,
            'F1_corrupted': f1_corr,
            'degradation_pp': drop_pp,
            'degradation_pct': drop_pct,
        })

degrad_df = pd.DataFrame(degrad_rows)
degrad_df.to_csv(OUTPUT_ROOT / 'robustness_results.csv', index=False)

print('=== Degradation Table ===')
display(degrad_df.round(3))

## 7. Visualizations

In [ ]:
# Grouped bar chart: Clean vs Corrupted F1 per field per pipeline
fields_to_plot = ['vendor', 'date', 'total', 'micro']
pipelines = results_df['pipeline'].unique()

fig, axes = plt.subplots(1, len(pipelines), figsize=(7*len(pipelines), 5), sharey=True)
if len(pipelines) == 1:
    axes = [axes]

for ax, pipe in zip(axes, pipelines):
    pipe_data = results_df[results_df['pipeline'] == pipe]
    x = np.arange(len(fields_to_plot))
    w = 0.35

    clean_f1 = []
    corr_f1 = []
    for f in fields_to_plot:
        c = pipe_data[(pipe_data['condition']=='Clean') & (pipe_data['field']==f)]
        clean_f1.append(c.iloc[0]['f1'] if len(c) else 0)
        c = pipe_data[(pipe_data['condition']=='Corrupted') & (pipe_data['field']==f)]
        corr_f1.append(c.iloc[0]['f1'] if len(c) else 0)

    bars1 = ax.bar(x - w/2, clean_f1, w, label='Clean', color='steelblue', alpha=0.85)
    bars2 = ax.bar(x + w/2, corr_f1, w, label='Corrupted', color='coral', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(fields_to_plot)
    ax.set_ylabel('F1 Score')
    ax.set_title(f'{pipe} Pipeline')
    ax.legend()
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.3)

    for bar in bars1 + bars2:
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.2f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Clean vs Corrupted F1 by Field', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Degradation bar chart
if len(degrad_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_data = degrad_df[degrad_df['field'] != 'micro']

    x = np.arange(len(plot_data))
    colors = ['steelblue' if p == 'EasyOCR' else 'orange' for p in plot_data['pipeline']]

    bars = ax.bar(x, plot_data['degradation_pp'], color=colors, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r['pipeline']}\n{r['field']}" for _, r in plot_data.iterrows()], fontsize=9)
    ax.set_ylabel('F1 Degradation (pp)')
    ax.set_title('Robustness: F1 Drop from Clean to Corrupted')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, plot_data['degradation_pp']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{val:.3f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print('No degradation data to plot.')

## Done

Results saved to `v2/outputs/robustness_results.csv`.
Proceed to 05_summary.ipynb for final comparison tables and visualizations.